In [2]:
import pandas as pd
from datasets import load_dataset

/scratch/s3799042/venvs/think-reduction/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/scratch/s3799042/venvs/think-reduction/lib/python3.10/site-packages/torch/cuda/__init__.py:63: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


In [3]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "MIG-0fb6bae2-fd70-5e49-921b-9d8c9f43e593" #"MIG-03c7a8b7-dd9c-5bda-82fb-1cf2e9c3a34d"
import torch
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM
import numpy as np
print(torch.cuda.device_count())
print(torch.cuda.get_device_name(0))

1
NVIDIA A100-SXM4-80GB MIG 3g.40gb


In [4]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model_path = "l3lab/L1-Qwen3-8B-Max"
model_LCPO = AutoModelForCausalLM.from_pretrained(
    model_path,
    torch_dtype=torch.bfloat16,
    device_map=device
)
tokenizer = AutoTokenizer.from_pretrained(model_path)
model_LCPO.eval()


`torch_dtype` is deprecated! Use `dtype` instead!
Loading checkpoint shards: 100%|██████████| 2/2 [00:03<00:00,  1.91s/it]


Qwen3ForCausalLM(
  (model): Qwen3Model(
    (embed_tokens): Embedding(151936, 4096)
    (layers): ModuleList(
      (0-35): 36 x Qwen3DecoderLayer(
        (self_attn): Qwen3Attention(
          (q_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear(in_features=4096, out_features=1024, bias=False)
          (v_proj): Linear(in_features=4096, out_features=1024, bias=False)
          (o_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
        )
        (mlp): Qwen3MLP(
          (gate_proj): Linear(in_features=4096, out_features=12288, bias=False)
          (up_proj): Linear(in_features=4096, out_features=12288, bias=False)
          (down_proj): Linear(in_features=12288, out_features=4096, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen3RMSNorm((4096,), eps=1e-06)
        (post_attention_la

In [5]:
def generate_hidden_states_cache(df, prompt_column_name="prompt", think_step_by_step=True):

    states = []
    cache = {}

    for _, q in tqdm(df.iterrows(), total=len(df), desc="Processing rows"):

        if think_step_by_step:
            prompt = f"{q[prompt_column_name]} Let’s think step by step inside and output the final answer within boxed{{}}."
        else:
            prompt = q[prompt_column_name]

        if prompt in cache:
            states.append(cache[prompt])
            continue

        inputs = tokenizer(prompt, return_tensors="pt").to(device)

        with torch.no_grad():
            outputs = model_LCPO(**inputs, output_hidden_states=True)
            hidden = outputs.hidden_states[-1][:, -1, :]

        hidden = (
            hidden.detach()
            .to(torch.float16)
            .cpu()
            .numpy()
            .reshape(-1)
        )

        cache[prompt] = hidden
        states.append(hidden)

        # free GPU memory
        del inputs, outputs
        torch.cuda.empty_cache()

    states = np.stack(states, axis=0)
    df["hidden"] = list(states)

    return df

In [9]:
from pathlib import Path
import pandas as pd

path_eval = Path().resolve().parent.parent / "processed" / "eval_L1_bert" / "gsm8k_olympiad_amc.parquet"
df_eval = pd.read_parquet(path_eval)

In [17]:
df_eval_hidden = generate_hidden_states_cache(df_eval, prompt_column_name = "prompt", think_step_by_step=True)

Processing rows: 100%|██████████| 2033/2033 [01:34<00:00, 21.52it/s]


In [ ]:
df_eval_hidden
cols_with_none = df_eval_hidden[df_eval_hidden['solution_col'].isna()]
cols_with_none

,id,dataset,prompt,solution_col,level,hidden
1340,1340,olympiad,"For each positive integer $k$, let $t(k)$ be t...",None,None,"[-1.039, 0.06104, -1.18, 1.086, -1.828, 1.789,..."
1356,1356,olympiad,Find all positive integers $n$ with the follow...,None,None,"[-1.023, -0.04248, -1.031, 0.668, -2.547, 3.04..."
1400,1400,olympiad,Find all positive integers $n \geqslant 2$ for...,None,None,"[-0.7773, 0.4258, -0.574, 0.8516, -2.688, 2.61..."
1405,1405,olympiad,Let $\mathbb{Z}_{\geqslant 0}$ be the set of n...,None,None,"[-0.66, 0.165, -0.4824, 1.195, -1.641, 1.633, ..."
1423,1423,olympiad,Find all positive integers $n$ such that there...,None,None,"[-0.836, 0.10254, -0.996, 0.4238, -2.172, 2.20..."
1436,1436,olympiad,"Let $m>1$ be an integer. A sequence $a_{1}, a_...",None,None,"[-0.3398, 0.3594, -0.789, 0.1953, -1.852, 1.85..."
1441,1441,olympiad,Determine all integers $m$ for which the $m \t...,None,None,"[0.3574, 1.273, -1.508, 0.7188, -2.812, 3.016,..."
1444,1444,olympiad,There are two increasing sequences of five con...,None,None,"[-0.6445, 1.195, -0.6055, 0.3418, -1.477, 2.92..."
1446,1446,olympiad,Determine all integer values of $x$ such that ...,None,None,"[-0.621, -0.1309, -0.3242, -0.8594, -1.875, 2...."
1450,1450,olympiad,"Determine, with justification, all values of $...",None,None,"[1.508, 1.164, 0.1357, -0.7812, -2.0, 2.484, 2..."


In [19]:
save_path = Path().resolve().parent.parent / "processed" /"eval_L1_bert"/ "gsm8k_olympiad_amc_hidden_L1.parquet"
df_eval_hidden.to_parquet(save_path)

In [10]:
df_targets_hidden.to_parquet(save_path)

In [12]:
#TEST
path_test_targets = Path().resolve().parent.parent / "old" / "dataset_splitting" / "test.parquet"
df_test_targets = pd.read_parquet(path_test_targets)

In [13]:
df_test_hidden = generate_hidden_states_cache(df_test_targets, prompt_column_name = "prompt", think_step_by_step=False)

Processing rows: 100%|██████████| 15000/15000 [00:39<00:00, 379.96it/s]


In [18]:
save_path_test = Path().resolve().parent.parent / "processed" /"train"/ "test_targets_hidden.parquet"
df_test_hidden.to_parquet(save_path_test)

BERT

In [13]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [14]:
from transformers import AutoTokenizer, AutoModel

tokenizer_bert = AutoTokenizer.from_pretrained(
    "bert-base-uncased",
    cache_dir="/scratch/s3799042/",
)

model_bert = AutoModel.from_pretrained(
    "bert-base-uncased",
    cache_dir="/scratch/s3799042/",
    torch_dtype=torch.float16,
    device_map="cuda",
)

In [15]:
from tqdm import tqdm
def generate_hidden_states_cache_bert(df, prompt_column_name="prompt", think_step_by_step=True):

    states = []
    cache = {}

    for _, q in tqdm(df.iterrows(), total=len(df), desc="Processing rows"):
        if think_step_by_step:
            prompt = f"{q[prompt_column_name]} Let’s think step by step inside and output the final answer within boxed{{}}."
        else:
            prompt = q[prompt_column_name]

        if prompt in cache:
            states.append(cache[prompt])
            continue

        enc = tokenizer_bert(
            prompt,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=512,
            add_special_tokens=True,
        )

        input_ids = enc["input_ids"].to(device)


        attention_mask = enc["attention_mask"].to(device)

        with torch.no_grad():
            out = model_bert(
                input_ids=input_ids,
                attention_mask=attention_mask,
            )

        cls_embeddings = out.last_hidden_state[:, 0, :]


        cache[prompt] = cls_embeddings.cpu().numpy().astype(np.float16).reshape(-1)
        states.append(cls_embeddings.cpu().numpy().astype(np.float16).reshape(-1))

        # free GPU memory
        #del inputs, outputs
        #torch.cuda.empty_cache()

    states = np.stack(states, axis=0)
    df["hidden"] = list(states)

    return df

In [16]:
df_eval_hidden_bert = generate_hidden_states_cache_bert(df_eval, prompt_column_name = "prompt", think_step_by_step=False)
save_path = Path().resolve().parent.parent / "processed" /"eval_L1_bert"/ "gsm8k_olympiad_amc_hidden_bert.parquet"
df_eval_hidden_bert.to_parquet(save_path)

Processing rows: 100%|██████████| 2033/2033 [00:12<00:00, 159.00it/s]


In [ ]:
from pathlib import Path
import pandas as pd

path_train_targets_bert = Path().resolve().parent.parent / "processed" / "dataset_splitting" / "train.parquet"
df_train_targets_bert = pd.read_parquet(path_train_targets_bert)


In [33]:
df_train_targets_hidden_bert = generate_hidden_states_cache_bert(df_train_targets_bert, prompt_column_name = "prompt", think_step_by_step=False)

Processing rows: 100%|██████████| 253660/253660 [01:31<00:00, 2783.95it/s]


In [34]:
df_train_targets_hidden_bert.to_parquet(Path().resolve().parent.parent / "processed" / "dataset_splitting" / "train_targets_hidden_bert.parquet")

In [31]:
path_test_targets_bert = Path().resolve().parent.parent / "processed" / "dataset_splitting" / "test.parquet"
df_test_targets_bert = pd.read_parquet(path_test_targets_bert)
df_test_targets_hidden_bert = generate_hidden_states_cache_bert(df_test_targets_bert, prompt_column_name = "prompt", think_step_by_step=False)
df_test_targets_hidden_bert

Processing rows: 100%|██████████| 15000/15000 [00:05<00:00, 2733.83it/s]


,question_id,prompt,solution_col,generated_think_text,generated_text,target_think_tokens,generated_think_tokens,latency_sec,is_correct,level,hidden
0,1254,"If $2^8=4^x$, what is the value of $x$? Let’s ...",Rewrite $4$ as $2^2$ to find $4^x=2^{2x}$. Si...,"Okay, 2^8 is 256. 4^x is (2^2)^x = 2^(2x). So ...","If $2^8=4^x$, what is the value of $x$? Let’s ...",100,53,0.906188,True,Level 5,"[-0.2898, -0.2683, 0.1577, -0.2142, -0.1013, -..."
1,1254,"If $2^8=4^x$, what is the value of $x$? Let’s ...",Rewrite $4$ as $2^2$ to find $4^x=2^{2x}$. Si...,"Okay, 2^8 equals 4^x. Since 4 is 2 squared, re...","If $2^8=4^x$, what is the value of $x$? Let’s ...",357,80,0.906188,True,Level 5,"[-0.2898, -0.2683, 0.1577, -0.2142, -0.1013, -..."
2,1254,"If $2^8=4^x$, what is the value of $x$? Let’s ...",Rewrite $4$ as $2^2$ to find $4^x=2^{2x}$. Si...,"Okay, I need to solve 2^8 = 4^x. Let me rewrit...","If $2^8=4^x$, what is the value of $x$? Let’s ...",615,164,0.906188,True,Level 5,"[-0.2898, -0.2683, 0.1577, -0.2142, -0.1013, -..."
3,1254,"If $2^8=4^x$, what is the value of $x$? Let’s ...",Rewrite $4$ as $2^2$ to find $4^x=2^{2x}$. Si...,"Okay, so I need to solve 2^8 equals 4^x. Hmm, ...","If $2^8=4^x$, what is the value of $x$? Let’s ...",873,257,0.906188,True,Level 5,"[-0.2898, -0.2683, 0.1577, -0.2142, -0.1013, -..."
4,1254,"If $2^8=4^x$, what is the value of $x$? Let’s ...",Rewrite $4$ as $2^2$ to find $4^x=2^{2x}$. Si...,"Okay, so I need to solve the equation 2^8 = 4^...","If $2^8=4^x$, what is the value of $x$? Let’s ...",1131,439,0.906188,True,Level 5,"[-0.2898, -0.2683, 0.1577, -0.2142, -0.1013, -..."
...,...,...,...,...,...,...,...,...,...,...,...
14995,aime_931,Let $b \geq 2$ be an integer. Call a positive ...,211,"Okay, so I need to find the smallest base b (a...",Let $b \geq 2$ be an integer. Call a positive ...,3968,2658,1.130943,False,aime,"[-0.3596, -0.4985, -0.109, -0.04092, -0.1237, ..."
14996,aime_931,Let $b \geq 2$ be an integer. Call a positive ...,211,"Okay, so I need to find the smallest base b (w...",Let $b \geq 2$ be an integer. Call a positive ...,4226,2607,1.130943,False,aime,"[-0.3596, -0.4985, -0.109, -0.04092, -0.1237, ..."
14997,aime_931,Let $b \geq 2$ be an integer. Call a positive ...,211,"Okay, so I need to find the smallest base b (w...",Let $b \geq 2$ be an integer. Call a positive ...,4484,3041,1.130943,False,aime,"[-0.3596, -0.4985, -0.109, -0.04092, -0.1237, ..."
14998,aime_931,Let $b \geq 2$ be an integer. Call a positive ...,211,"Okay, so I need to find the smallest base b (w...",Let $b \geq 2$ be an integer. Call a positive ...,4742,2568,1.130943,False,aime,"[-0.3596, -0.4985, -0.109, -0.04092, -0.1237, ..."


In [32]:
df_test_targets_hidden_bert.to_parquet(Path().resolve().parent.parent / "processed" / "dataset_splitting" / "test_targets_hidden_bert.parquet")